In [ ]:
# ==================== SummarizationMiddleware 完整实现 ====================

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import ensure_config
from pydantic import BaseModel, Field
from typing import Optional
from dotenv import load_dotenv
import logging

# ==================== 1. 配置日志 ====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
load_dotenv(override=True)

# ==================== 2. 定义工具 ====================
@tool
def search_patent(query: str) -> str:
    """搜索专利数据库"""
    return f"专利搜索结果: 找到与 '{query}' 相关的 3 项专利..."

@tool
def analyze_technology(tech_desc: str) -> str:
    """分析技术可行性"""
    return f"技术分析: '{tech_desc}' 的实现可行性评估完成..."

tools = [search_patent, analyze_technology]

# ==================== 3. 定义上下文 ====================
class UserContext(BaseModel):
    user_id: str = Field(..., description="用户唯一标识")
    department: str = Field(..., description="所属部门")
    max_history_tokens: Optional[int] = Field(default=1000, description="历史消息 token 阈值")

# ==================== 4. 配置中间件 ====================
summarization_middleware = SummarizationMiddleware(
    model=ChatDeepSeek(model="deepseek-chat", temperature=0.1),
    # max_tokens_before_summary=200,          # 历史消息 token 数量超过 200 时触发压缩
    messages_to_keep=5,                     # 保留最近 5 条消息
    summary_prompt="请将以下对话历史进行摘要，保留关键决策点和技术细节：\n\n{messages}\n\n摘要:"   # 摘要提示词
)

# ==================== 5. 创建 Agent ====================
agent = create_agent(
    model=ChatDeepSeek(model="deepseek-chat", temperature=0.2),
    tools=tools,
    middleware=[summarization_middleware],
    context_schema=UserContext,
    debug=True,
)

# ==================== 6. 执行测试 ====================
def run_summarization_test():
    logger.info("开始 SummarizationMiddleware 测试")

    # 创建长对话历史
    long_history = [HumanMessage(content=f"问题 {i+1}: 如何评估某项技术的专利风险？") for i in range(20)]
    logger.info(f"创建了 {len(long_history)} 条消息")

    # 执行
    result = agent.invoke(
        {"messages": long_history},
        context=UserContext(user_id="engineer_001", department="研发部"),
        config=ensure_config({"configurable": {"thread_id": "session_001"}})
    )

    result_messages = result.get("messages", [])
    logger.info(f"执行后消息数: {len(result_messages)}")

    if len(result_messages) < len(long_history):
        logger.info(f"中间件已触发！压缩了 {len(long_history) - len(result_messages)} 条消息")

    return result

# ==================== 7. 运行测试 ====================
result = run_summarization_test()
logger.info("测试完成")

/data/data/com.termux/files/usr/tmp/ipykernel_17911/3570423299.py:39: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  summarization_middleware = SummarizationMiddleware(
2026-05-19 18:58:30,639 - INFO - 开始 SummarizationMiddleware 测试
2026-05-19 18:58:30,641 - INFO - 创建了 20 条消息
2026-05-19 18:58:30,794 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[values] {'messages': [HumanMessage(content='问题 1: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='20fa2351-10c4-4e5b-b5ca-5386b93baef1'), HumanMessage(content='问题 2: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='bacbb717-476e-4ee2-9d09-999e54324f20'), HumanMessage(content='问题 3: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='4213bf39-08eb-4a50-823f-56b12c695141'), HumanMessage(content='问题 4: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='c04df86c-0a80-4fa5-a564-a72dfbbf443d'), HumanMessage(content='问题 5: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='c1d5f4a5-9394-4f01-a49a-1845dbbb6caa'), HumanMessage(content='问题 6: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='af162e5c-9306-4e6c-aa55-5327a48a6d8b'), HumanMessage(content='问题 7: 如何评估某项技术的专利风险？', additional_kwargs={}, response_metadata={}, id='1252acc0-a220-42fe-898c-8b912b426cc9'), HumanMessage(content='问题 8: 如何评估某项技术的专利风险？', a

2026-05-19 18:58:31,493 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
2026-05-19 18:58:40,811 - INFO - 执行后消息数: 21
2026-05-19 18:58:40,813 - INFO - 测试完成


[updates] {'model': {'messages': [AIMessage(content='看起来您重复问了同一个问题20次。我来为您系统性地解答：**如何评估某项技术的专利风险？**\n\n这是一个非常重要的问题，以下是完整的评估框架：\n\n---\n\n## 专利风险评估的完整流程\n\n### 一、专利检索与清查（Patent Clearance Search）\n\n1. **确定技术特征**：拆解技术的核心要素、功能模块、实现方法\n2. **构建检索策略**：\n   - 关键词（中英文同义词、近义词）\n   - 专利分类号（IPC、CPC、LOC）\n   - 申请人/发明人\n   - 时间范围\n3. **多数据库检索**：中国专利数据库、USPTO、EPO、WIPO等\n\n### 二、专利筛选与标引\n\n- 阅读专利摘要、权利要求、说明书\n- 标记相关专利（高、中、低相关度）\n- 识别**有效专利**（未过期、维持有效）\n\n### 三、侵权分析（Claim Chart分析）\n\n| 步骤 | 内容 |\n|------|------|\n| 1 | 将技术特征与专利权利要求逐项比对 |\n| 2 | 判断是否落入**字面侵权**（Literal Infringement） |\n| 3 | 判断是否适用**等同原则**（Doctrine of Equivalents） |\n| 4 | 评估是否存在**现有技术抗辩**或**豁免情形** |\n\n### 四、专利有效性分析\n\n- 评估对方专利是否可能被**无效宣告**\n- 检查：新颖性、创造性、实用性、充分公开\n- 检索现有技术以挑战专利有效性\n\n### 五、风险等级评估\n\n| 风险等级 | 描述 | 应对策略 |\n|---------|------|---------|\n| 🔴 高风险 | 明确落入有效专利权利要求范围 | 规避设计、获取许可、无效宣告 |\n| 🟡 中风险 | 可能落入等同范围或专利有效性存疑 | 深入分析、准备抗辩方案 |\n| 🟢 低风险 | 未落入或专利已失效 | 监控更新、定期复查 |\n\n### 六、FTO（自由实施）报告\n\n- 出具正式的**Freedom to Operate**分析报告\n- 记录分析过

2026-05-19 18:58:41,635 - WARNING - Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
